In [1]:
import boto3
import zipfile
import os

BUCKET_NAME = 'projekt-ml-odpady' 
FILE_NAME = 'archive.zip'

s3 = boto3.client('s3')

print("Pobieranie pliku z S3...")
s3.download_file(BUCKET_NAME, FILE_NAME, FILE_NAME)

print("Rozpakowywanie... (to może potrwać chwilę)")
with zipfile.ZipFile(FILE_NAME, 'r') as zip_ref:
    zip_ref.extractall('data')

print("Gotowe! Dane są w folderze 'data'.")

Pobieranie pliku z S3...
Rozpakowywanie... (to może potrwać chwilę)
Gotowe! Dane są w folderze 'data'.


In [1]:
%pip install gradio tensorflow matplotlib opencv-python pillow ipywidgets

  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached ml_dtypes-0.5.4-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (118 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (5.1 

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt

# 1. Przygotowanie danych
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'data/garbage classification/Garbage classification' 

train_dataset = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"Klasy: {class_names}")

# 2. Budowa modelu (Transfer Learning)
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                               include_top=False,
                                               weights='imagenet')
base_model.trainable = False # Zamrażamy bazę

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 3. Trening
print("Start treningu...")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5)

# 4. Wykresy 
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Dokładność modelu')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Strata modelu')
plt.show()

Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
Klasy: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Start treningu...
Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.4238 - loss: 1.4980 - val_accuracy: 0.5426 - val_loss: 1.1494
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.6019 - loss: 1.0694 - val_accuracy: 0.6238 - val_loss: 1.0121
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.6627 - loss: 0.9554 - val_accuracy: 0.6317 - val_loss: 0.9590
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 138s 1s/step - accuracy: 0.6845 - loss: 0.8773 - val_accuracy: 0.6673 - val_loss: 0.9144
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.7132 - loss: 0.8248 - val_accuracy: 0.6673 - val_loss: 0.8960


<Figure size 1000x400 with 2 Axes>

In [9]:
model.save('model_odpady_final.h5')
print("Model zapisany!")

Model zapisany! Możesz go teraz pobrać z panelu po lewej (odśwież listę plików).


In [2]:
import tensorflow as tf
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# 1. Definiujemy klasy ręcznie (żeby nie musieć ładować całego datasetu)
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# 2. Wczytujemy gotowy model
model = tf.keras.models.load_model('model_odpady_final.h5')

print("Model i klasy są gotowe do pracy.")

I0000 00:00:1783167936.583549    5417 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
E0000 00:00:1783167949.979273    5417 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Model i klasy są gotowe do pracy.


In [4]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
import cv2

# --- FUNKCJA GRAD-CAM (WERSJA "ULTRA-STABILNA") ---
def get_gradcam_heatmap(img_array, model):
    try:
        base_model = model.layers[0]
        
        target_layer = None
        for layer in reversed(base_model.layers):
            if 'conv' in layer.name.lower():
                target_layer = layer
                break
        
        if target_layer is None: 
            print("Nie znaleziono warstwy konwolucyjnej.")
            return None

        feature_extractor = tf.keras.Model(inputs=base_model.inputs, outputs=target_layer.output)
        
        with tf.GradientTape() as tape:
            # Wyciągamy cechy z obrazka
            features = feature_extractor(img_array)
            tape.watch(features)
            
            current_output = features
        
            gap_features = tf.reduce_mean(features, axis=(1, 2))
            # Przechodzimy przez ostatnią warstwę Dense (model.layers[2])
            predictions = model.layers[2](gap_features)
            
            pred_index = tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]

        # Gradienty klasy względem mapy cech
        grads = tape.gradient(class_channel, features)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        features = features[0]
        heatmap = features @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

        heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
        return heatmap.numpy()
        
    except Exception as e:
        print(f"Szczegółowy błąd Grad-CAM: {e}")
        return None

def predict_and_visualize(img):
    if img is None: return None, None
    
    img_pil = Image.fromarray(img.astype('uint8'), 'RGB')
    img_224 = img_pil.resize((224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img_224)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    # 1. Wynik klasyfikacji
    preds = model.predict(img_array)
    label = {class_names[i]: float(preds[0][i]) for i in range(len(class_names))}

    # 2. Wizualizacja (Mapa Ciepła)
    heatmap = get_gradcam_heatmap(img_array, model)
    
    original_img = np.array(img)
    if heatmap is not None:
        # Nakładanie mapy kolorów
        heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
        heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        
        # Mieszanie: 60% zdjęcia + 40% koloru
        viz_img = cv2.addWeighted(original_img, 0.6, heatmap_color, 0.4, 0)
    else:
        # Jeśli się nie uda, pokaż chociaż mały czerwony róg (żebyś wiedziała, że kod żyje)
        viz_img = original_img.copy()
        cv2.rectangle(viz_img, (0,0), (50,50), (0,0,255), -1)
        
    return label, viz_img

# --- INTERFEJS ---
demo = gr.Interface(
    fn=predict_and_visualize,
    inputs=gr.Image(sources=["webcam"], label="Aparat"),
    outputs=[
        gr.Label(num_top_classes=3, label="Klasyfikacja"),
        gr.Image(label="Gdzie patrzy AI (Czerwony = Najważniejsze)")
    ],
    title="♻️ System Klasyfikacji Odpadów + Analiza XAI",
    description="Wytrenowano na MobileNetV2. Mapa ciepła udowadnia, że model patrzy na cechy fizyczne przedmiotu."
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://cb1b10ff6fe4eb16ab.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
